In [1]:
import requests
import pandas as pd
import time

In [2]:
with open("../data/raw/app_ids_large_seed.txt", "r") as file:
    app_ids = [int(line.strip()) for line in file if line.strip()]

print(len(app_ids))
print(app_ids[:10])

483
[10, 20, 30, 40, 50, 60, 70, 80, 130, 220]


In [3]:
url = f"https://store.steampowered.com/api/appdetails?appids={730}"
response = requests.get(url, timeout=10)
data = response.json()
print(data.keys())

dict_keys(['730'])


In [4]:
data['730']

{'success': True,
 'data': {'type': 'game',
  'name': 'Counter-Strike 2',
  'steam_appid': 730,
  'required_age': 0,
  'is_free': True,
  'dlc': [2678630],
  'detailed_description': 'For over two decades, Counter-Strike has offered an elite competitive experience, one shaped by millions of players from across the globe. And now the next chapter in the CS story is about to begin. This is Counter-Strike 2.<br><br>A free upgrade to CS:GO, Counter-Strike 2 marks the largest technical leap in Counter-Strike’s history. Built on the Source 2 engine, Counter-Strike 2 is modernized with realistic physically-based rendering, state of the art networking, and upgraded Community Workshop tools.<br><br>In addition to the classic objective-focused gameplay that Counter-Strike pioneered in 1999, Counter-Strike 2 features:<br><br><ul class="bb_ul"><li>All-new CS Ratings with the updated Premier mode<br></li><li>Global and Regional leaderboards<br></li><li>Upgraded and overhauled maps<br></li><li>Game-c

In [6]:
def get_game_metadata(app_id):
    url = f"https://store.steampowered.com/api/appdetails?appids={app_id}"
    
    try:
        response = requests.get(url, timeout=10)
        data = response.json()
        
        if not data[str(app_id)]["success"]:
            return None
        
        game = data[str(app_id)]["data"]
        
        return {
            "app_id": app_id,
            "name": game.get("name"),
            "type": game.get("type"),
            "is_free": game.get("is_free"),
            "release_date": game.get("release_date", {}).get("date"),
            "coming_soon": game.get("release_date", {}).get("coming_soon"),
            "price_usd": (
                game.get("price_overview", {}).get("final", None) / 100
                if game.get("price_overview") and game.get("price_overview", {}).get("final") is not None
                else 0 if game.get("is_free") else None
            ),
            "developers": ", ".join(game.get("developers", [])) if game.get("developers") else None,
            "publishers": ", ".join(game.get("publishers", [])) if game.get("publishers") else None,
            "genres": ", ".join([g["description"] for g in game.get("genres", [])]) if game.get("genres") else None,
            "categories": ", ".join([c["description"] for c in game.get("categories", [])]) if game.get("categories") else None,
            "dlc_count": len(game.get("dlc", [])) if game.get("dlc") else 0,
            "required_age": game.get("required_age"),
            "header_image": game.get("header_image"),
            "website": game.get("website")
        }
    
    except Exception as e:
        print(f"Error for app_id {app_id}: {e}")
        return None

In [7]:
records = []

for app_id in app_ids:
    row = get_game_metadata(app_id)
    if row is not None:
        records.append(row)
    time.sleep(1)  # give the API some time

df_metadata = pd.DataFrame(records)
df_metadata.head()

,app_id,name,type,is_free,release_date,coming_soon,price_usd,developers,publishers,genres,categories,dlc_count,required_age,header_image,website
0,10,Counter-Strike,game,False,"Nov 1, 2000",False,9.99,Valve,Valve,Action,"Multi-player, PvP, Online PvP, Shared/Split Sc...",0,0,https://shared.akamai.steamstatic.com/store_it...,None
1,20,Team Fortress Classic,game,False,"1 Apr, 1999",False,7.50,Valve,Valve,Action,"Multi-player, PvP, Online PvP, Shared/Split Sc...",0,0,https://shared.akamai.steamstatic.com/store_it...,None
2,30,Day of Defeat,game,False,"May 1, 2003",False,4.99,Valve,Valve,Action,"Multi-player, Camera Comfort, Color Alternativ...",0,0,https://shared.akamai.steamstatic.com/store_it...,http://www.dayofdefeat.com/
3,40,Deathmatch Classic,game,False,"Jun 1, 2001",False,4.99,Valve,Valve,Action,"Multi-player, PvP, Online PvP, Shared/Split Sc...",0,0,https://shared.akamai.steamstatic.com/store_it...,None
4,50,Half-Life: Opposing Force,game,False,"Nov 1, 1999",False,4.99,Gearbox Software,Valve,Action,"Single-player, Multi-player, Custom Volume Con...",0,0,https://shared.akamai.steamstatic.com/store_it...,None


In [8]:
print(df_metadata.shape)
print(df_metadata.columns)
df_metadata.info()

(480, 15)
Index(['app_id', 'name', 'type', 'is_free', 'release_date', 'coming_soon',
       'price_usd', 'developers', 'publishers', 'genres', 'categories',
       'dlc_count', 'required_age', 'header_image', 'website'],
      dtype='object')
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 480 entries, 0 to 479
Data columns (total 15 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   app_id        480 non-null    int64  
 1   name          480 non-null    object 
 2   type          480 non-null    object 
 3   is_free       480 non-null    bool   
 4   release_date  480 non-null    object 
 5   coming_soon   480 non-null    bool   
 6   price_usd     441 non-null    float64
 7   developers    480 non-null    object 
 8   publishers    478 non-null    object 
 9   genres        477 non-null    object 
 10  categories    479 non-null    object 
 11  dlc_count     480 non-null    int64  
 12  required_age  480 non-null    object 
 13  he

In [9]:
df_metadata.to_csv("../data/raw/steam_metadata_batch1.csv", index=False)
print("Saved first batch.")

Saved first batch.
